# 1. Preparation   
This is a Python Notebook.  
To make it work, you need to press "play" on the code cells.   
Remember to always run cells in the right order and never skip one!   
In case of doubt, you can always restart from the beginning.   

First, let's clone the course repository so we can access our curated dataset of annotated passages (YES / NO).

In [ ]:
# Clone the course repository to access the dataset
!git clone https://github.com/SimoneRebora/CMCLS

In [ ]:
# Import the pandas library (Python's tool for working with tables/spreadsheets)
import pandas as pd

# Read the CSV file directly from the cloned CMCLS repository
my_df = pd.read_csv('/content/CMCLS/materials/HoD_annotated_dataset.csv', encoding='utf-8', encoding_errors='replace')

# Remove any accidental empty rows where 'Sentences' or 'curation' is missing
my_df = my_df.dropna(subset=['Sentences', 'curation']).copy()

# Show the dataset table on screen
my_df

2. Machine Learning:

Before training the computer, we divide our dataset into two parts:

Train (80%): The study material the computer learns from (with the answers).

Test (20%): The hidden final exam we use to check if the computer actually learned.

In [ ]:
# Import the function that splits our table into train and test sets
from sklearn.model_selection import train_test_split

# Split 'my_df' into 80% training data and 20% testing data
# random_state=123 ensures the random shuffle is repeatable and matches Simone's split
train, test = train_test_split(my_df, test_size=0.2, random_state=123)

# Print how many passages are in each pile
print("Training passages:", len(train))
print("Testing passages:", len(test))

#Create word frequency features

We use CountVectorizer to turn each passage into a matrix counting how many times each vocabulary word appears.

In [ ]:
# Import CountVectorizer to convert text into word frequency counts
from sklearn.feature_extraction.text import CountVectorizer

# Set up the word counter:
# - lowercase=True: treats "Other" and "other" as the same word
# - stop_words='english': ignores common filler words like "the", "and", "is"
# - min_df=2: ignores ultra-rare words that appear in fewer than 2 passages
vectorizer = CountVectorizer(lowercase=True, stop_words='english', min_df=2)

# Learn the vocabulary from the training passages and count word frequencies (train_X)
train_X = vectorizer.fit_transform(train['Sentences'])
# Save the true YES/NO answers for the training passages (train_y)
train_y = train['curation']

# Count the same vocabulary words in the hidden test passages (test_X)
# Note: We use .transform() (not .fit_transform()) so the model doesn't peek at the exam!
test_X = vectorizer.transform(test['Sentences'])
# Save the true YES/NO answers for the test passages (test_y)
test_y = test['curation']

# Print the dimensions of our number tables: (number of passages, number of unique words)
print("Train table shape:", train_X.shape)
print("Test table shape:", test_X.shape)

#Logistic regression
Train a logistic regression model with word frequencies (to predict YES vs. NO).

In [ ]:
# Import the Logistic Regression classifier
from sklearn.linear_model import LogisticRegression

# Create an empty Logistic Regression model
# max_iter=1000 gives the math algorithm plenty of steps to find the best word weights
model = LogisticRegression()

# Train ("fit") the model by showing it the training word counts (train_X) and answers (train_y)
model.fit(train_X, train_y)

#Make predictions with the model on the test set
Now we hide the true answers on the 20% test set and ask our trained model to guess YES or NO for each passage.

In [ ]:
# Ask the trained model to predict YES or NO for the test passages
prediction = model.predict(test_X)

# Add the model's guesses as a new column in our 'test' table so we can compare side-by-side
test['logistic regression'] = prediction

# Display the test table with true labels and model predictions
test

#See the efficiency scores
Let's print the official report card comparing the human annotations (test_y) against the model's guesses (prediction).

In [ ]:
# Import the classification report tool to grade the model's predictions
from sklearn.metrics import classification_report

# Print precision, recall, and f1-score comparing true answers vs. model predictions
print(classification_report(test_y, prediction))

#Peek inside the model: Which words mattered most?
The top 10 words that pushed the model to guess YES versus NO.

In [ ]:
# Import numpy to help sort the word weights
import numpy as np

# Get the list of all vocabulary words
words = np.array(vectorizer.get_feature_names_out())
# Get the importance weight ("clue score") the model assigned to each word
coefs = model.coef_[0]

# Print the 10 words with the highest positive scores (pushing toward YES)
print("Top 10 words pushing toward YES:")
for word, score in zip(words[np.argsort(coefs)[-10:][::-1]], np.sort(coefs)[-10:][::-1]):
    print(f"  {word:<15} {score:.3f}")

# Print the 10 words with the lowest negative scores (pushing toward NO)
print("\nTop 10 words pushing toward NO:")
for word, score in zip(words[np.argsort(coefs)[:10]], np.sort(coefs)[:10]):
    print(f"  {word:<15} {score:.3f}")